In [11]:
import os
import glob

import json
import numpy as np
import pandas as pd

from pathlib import Path
from typing import List, Tuple, Any, Optional
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

from concurrent.futures import ProcessPoolExecutor, as_completed

In [12]:
# STORAGE_DIR = "Z:\\starcraft-vision\\"
STORAGE_DIR = "/workspace"

DATA_DIR = os.path.join(STORAGE_DIR, "data")
GROUND_TRUTH_DIR = os.path.join(DATA_DIR, "label", "dst")
PREDICTIONS_DIR = os.path.join(STORAGE_DIR, "predictions")

In [13]:
# REPLAYS = [36, 212, 438, 522, 1660, 6254]
# REPLAYS = [1559, 1628, 2351, 6219, 11251]
# REPLAYS = [275, 1725, 3613, 4520, 4664]
REPLAYS = [36, 212, 438, 522, 1660, 1559, 1628, 2351, 6219, 11251, 275, 1725, 3613, 4520, 4664]

REPLAYS = [str(_) + '.rep' for _ in REPLAYS]

In [14]:
# TARGET_MODEL = "test_input_model/model_020"
# TARGET_MODEL = "all_correct_win4_b16_20250826_045543/model_030"
TARGET_MODEL_1 = "all_correct_win4_b16_20250826_045543/model_030"
TARGET_MODEL_2 = "all_correct_win4_b16_kbrs_20250912_005403/model_030"


In [15]:
ground_truth_json_path = [
    f for name in REPLAYS
    for f in glob.glob(os.path.join(GROUND_TRUTH_DIR, name, '*.json'))
]

print(f"Found {len(ground_truth_json_path)} ground truth files.")

Found 15 ground truth files.


In [26]:
import os, json, mmap
from pathlib import Path
from typing import Any, List, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

# ---------- 경로 수집 (빠른 scandir 재귀) ----------
def _iter_json_paths(root: str) -> List[str]:
    out = []
    stack = [os.path.abspath(root)]
    while stack:
        d = stack.pop()
        try:
            with os.scandir(d) as it:
                for e in it:
                    if e.is_dir(follow_symlinks=False):
                        stack.append(e.path)
                    elif e.is_file() and e.name.endswith(".json"):
                        out.append(e.path)
        except PermissionError:
            pass
    return out

def _build_json_paths(directory: str,
                      label_method: str = "all_correct",
                      replays: Optional[List[str]] = None) -> List[str]:
    directory = os.path.abspath(directory)
    if replays:
        paths = [os.path.join(directory, rep, f"{label_method}.json") for rep in replays]
        # 존재하는 파일만
        return [p for p in paths if os.path.isfile(p)]
    # 전체 스캔
    return _iter_json_paths(directory)

# ---------- 빠른 로더 (orjson + mmap, fallback 표준 json) ----------
import os, json, mmap, gzip

def _read_one_json(path: str, use_orjson: bool = True):
    try:
        size = os.path.getsize(path)
        if size == 0:
            return path, None, "EmptyFileError: file size is 0"

        if use_orjson:
            try:
                import orjson
                with open(path, "rb") as f:
                    # gzip 여부 감지 (0x1f 0x8b)
                    head = f.read(2)
                    f.seek(0)
                    if head == b"\x1f\x8b":
                        with gzip.GzipFile(fileobj=f) as gz:
                            b = gz.read()  # bytes
                            return path, orjson.loads(b), None
                    # mmap + memoryview 로 zero-copy 파싱
                    with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:
                        return path, orjson.loads(memoryview(mm)), None
            except ModuleNotFoundError:
                pass  # orjson 없으면 표준 json으로 폴백

        # 표준 json 폴백 (BOM 처리 위해 utf-8-sig)
        with open(path, "r", encoding="utf-8-sig") as f:
            return path, json.load(f), None

    except Exception as e:
        return path, None, f"{type(e).__name__}: {e}"


# ---------- 자동 전략 선택 ----------
def _choose_strategy(paths: List[str], bytes_threshold: int = 512_000) -> str:
    """
    파일 크기 중앙값이 threshold 이상이면 'process', 아니면 'thread'
    - 작은 파일: 스레드가 유리(I/O 병목 + orjson이 GIL 거의 점유 안함)
    - 큰 파일: 프로세스가 유리(파싱 CPU 부담 분산)
    """
    if not paths:
        return "thread"
    sizes = []
    step = max(1, len(paths)//200)  # 샘플링
    for p in paths[::step]:
        try:
            sizes.append(os.path.getsize(p))
        except OSError:
            pass
    if not sizes:
        return "thread"
    from statistics import median
    return "process" if median(sizes) >= bytes_threshold else "thread"

# ---------- 공통 실행기 ----------
def read_json_files_fast(directory: str,
                         label_method: str = "all_correct",
                         replays: Optional[List[str]] = None,
                         max_workers: Optional[int] = None,
                         force: Optional[str] = None  # 'thread'|'process'|None(auto)
                         ) -> List[Any]:
    paths = _build_json_paths(directory, label_method, replays)
    if not paths:
        return []

    # 전략/워커 수 결정
    strategy = force or _choose_strategy(paths)
    if max_workers is None:
        cpu = os.cpu_count() or 4
        if strategy == "process":
            max_workers = min(cpu, 8)            # 과도한 프로세스는 오버헤드↑
        else:
            max_workers = min(32, cpu * 2)       # 작은 파일 다수에 적절

    results: List[Any] = []
    errors: List[str] = []

    if strategy == "process":
        # ProcessPool은 chunksize가 중요 (작은 태스크 묶음)
        chunksize = max(1, len(paths) // (max_workers * 8))
        with ProcessPoolExecutor(max_workers=max_workers) as ex:
            for path, data, err in ex.map(_read_one_json, paths, chunksize=chunksize):
                if err:
                    errors.append(f"[FAIL] {path} -> {err}")
                else:
                    results.append(data)
    else:
        # ThreadPool: orjson이 C에서 돌며 GIL을 잘 놓기 때문에 효율적
        with ThreadPoolExecutor(max_workers=max_workers, thread_name_prefix="json") as ex:
            for path, data, err in ex.map(_read_one_json, paths):
                if err:
                    errors.append(f"[FAIL] {path} -> {err}")
                else:
                    results.append(data)

    if errors:
        print("\n".join(errors))
    return results


In [27]:
ground_truth_data = read_json_files_fast(
    GROUND_TRUTH_DIR,
    label_method='all_correct', replays=REPLAYS,
    max_workers=16,
    force='process'
)

In [28]:
# 작은 JSON 다수면 thread가, 큰 JSON(수 MB+)이면 process가 유리
prediction_vanilla_data = read_json_files_fast(
    os.path.join(PREDICTIONS_DIR, TARGET_MODEL_1),
    label_method='all_correct', replays=REPLAYS,
    max_workers=16,             
    force='process'
)
prediction_kbrs_data = read_json_files_fast(
    os.path.join(PREDICTIONS_DIR, TARGET_MODEL_2),
    label_method='all_correct', replays=REPLAYS,
    max_workers=16,
    force='process'
)

In [29]:
def get_annotations(data, replay_suffix=".rep"):
    frames = []

    for item in (data or []):
        anns = item.get("annotations", [])
        if not anns:
            continue

        df = pd.DataFrame(anns)

        desc = (item.get("info") or {}).get("description", "") or ""
        replay_id = None
        if isinstance(desc, str) and desc.strip():
            replay_id = desc.strip().split()[-1]

        if replay_id is None:
            replay_id = "unknown"

        if replay_suffix and not str(replay_id).endswith(replay_suffix):
            replay_id = f"{replay_id}{replay_suffix}"

        df["replay"] = replay_id
        frames.append(df)

    base_cols = ["replay", "id", "image_id", "category_id", "bbox", "area", "segmentation", "iscrowd"]
    if not frames:
        return pd.DataFrame(columns=base_cols + ["score"])

    has_score = any("score" in f.columns for f in frames)

    wanted_cols = base_cols + (["score"] if has_score else [])
    for i, f in enumerate(frames):
        for col in wanted_cols:
            if col not in f.columns:
                frames[i][col] = pd.NA

    annotations = pd.concat(frames, ignore_index=True)

    return annotations[wanted_cols]

In [30]:
annotation_ground_truth = get_annotations(ground_truth_data)
annotation_prediction_vanilla = get_annotations(prediction_vanilla_data)
annotation_prediction_kbrs = get_annotations(prediction_kbrs_data)

In [31]:
def dataframe_topk_score(dataframe, K=1):
    topk_df = (
      dataframe.sort_values('score', ascending=False)
        .groupby(['replay', 'image_id'], group_keys=False)
        .head(K)
    )
    return topk_df.sort_values(['replay', 'image_id'])

In [32]:
top1_annotation_prediction_vanilla = dataframe_topk_score(annotation_prediction_vanilla, K=1)
top1_annotation_prediction_kbrs = dataframe_topk_score(annotation_prediction_kbrs, K=1)

In [33]:
annotation_ground_truth["idx"] = annotation_ground_truth.groupby(["replay", "image_id"]).cumcount()  # 0~4 인덱스 생성
wide_ground_truth = annotation_ground_truth.pivot(index=["replay", "image_id"], columns="idx", values="bbox")
wide_ground_truth.columns = [f"gt_{i}" for i in wide_ground_truth.columns]
wide_ground_truth.reset_index(inplace=True)


In [34]:
top1_annotation_prediction_vanilla["idx"] = top1_annotation_prediction_vanilla.groupby(["replay", "image_id"]).cumcount()  # 0~4 인덱스 생성
wide_prediction_vanilla = top1_annotation_prediction_vanilla.pivot(index=["replay", "image_id"], columns="idx", values="bbox")
wide_prediction_vanilla.columns = [f"pred_{i}" for i in wide_prediction_vanilla.columns]
wide_prediction_vanilla.reset_index(inplace=True)

top1_annotation_prediction_kbrs["idx"] = top1_annotation_prediction_kbrs.groupby(["replay", "image_id"]).cumcount()  # 0~4 인덱스 생성
wide_prediction_kbrs = top1_annotation_prediction_kbrs.pivot(index=["replay", "image_id"], columns="idx", values="bbox")
wide_prediction_kbrs.columns = [f"pred_{i}" for i in wide_prediction_kbrs.columns]
wide_prediction_kbrs.reset_index(inplace=True)

In [35]:
result_inference_vanilla = pd.merge(wide_ground_truth, wide_prediction_vanilla,on=["replay", "image_id"], how="outer")
result_inference_vanilla = result_inference_vanilla.bfill()
result_inference_vanilla = result_inference_vanilla.dropna()

new_cols = []

for c in result_inference_vanilla.columns:
    if c in ('replay', 'image_id'):
        new_cols.append(('meta', c))
    elif c.startswith('gt_'):
        new_cols.append(('gt', c.split('_')[1]))
    elif c.startswith('pred_'):
        new_cols.append(('pred', c.split('_')[1]))
    else:
        new_cols.append(('other', c))  # fallback

result_inference_vanilla.columns = pd.MultiIndex.from_tuples(new_cols)

result_inference_vanilla.head()

meta                         gt                                      \
      replay image_id                 0                 1                 2   
0  11251.rep        0  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
1  11251.rep        1  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
2  11251.rep        2  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
3  11251.rep        3  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
4  11251.rep        4  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   

                                                     pred  
                  3                 4                   0  
0  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 113, 20, 12]  
1  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 113, 20, 12]  
2  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 113, 20, 12]  
3  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 113, 20, 12]  
4  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 113, 20, 12]

In [36]:
result_inference_kbrs = pd.merge(wide_ground_truth, wide_prediction_kbrs,on=["replay", "image_id"], how="outer")
result_inference_kbrs = result_inference_kbrs.bfill()
result_inference_kbrs = result_inference_kbrs.dropna()

new_cols = []
for c in result_inference_kbrs.columns:
    if c in ('replay', 'image_id'):
        new_cols.append(('meta', c))
    elif c.startswith('gt_'):
        new_cols.append(('gt', c.split('_')[1]))
    elif c.startswith('pred_'):
        new_cols.append(('pred', c.split('_')[1]))
    else:
        new_cols.append(('other', c))  # fallback
        
result_inference_kbrs.columns = pd.MultiIndex.from_tuples(new_cols)
result_inference_kbrs.head()

meta                         gt                                      \
      replay image_id                 0                 1                 2   
0  11251.rep        0  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
1  11251.rep        1  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
2  11251.rep        2  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
3  11251.rep        3  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
4  11251.rep        4  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   

                                                 pred  
                  3                 4               0  
0  [108, 2, 20, 12]  [108, 2, 20, 12]  [0, 2, 19, 12]  
1  [108, 2, 20, 12]  [108, 2, 20, 12]  [0, 2, 19, 12]  
2  [108, 2, 20, 12]  [108, 2, 20, 12]  [0, 2, 19, 12]  
3  [108, 2, 20, 12]  [108, 2, 20, 12]  [0, 2, 19, 12]  
4  [108, 2, 20, 12]  [108, 2, 20, 12]  [0, 2, 19, 12]

In [37]:
def union_metrics_singlepass(rows, gt_cols, pred_cols,
                             grid_w, grid_h, orig_w, orig_h,
                             kernel_x, kernel_y, use_bbox_size, thresholds,
                             return_summary: bool = False):
    n = len(rows)
    ic_multi = np.full(n, np.nan, dtype=float)
    ic_ratio = np.zeros(n, dtype=float)
    ic_flags = {t: np.zeros(n, dtype=float) for t in thresholds}

    def xywh_to_rect(x,y,w,h):
        if not use_bbox_size:
            w,h = kernel_x, kernel_y
        rx = int(round(x * 32 / orig_w * (grid_w - w)))
        ry = int(round(y * 32 / orig_h * (grid_h - h)))
        rx = max(0, min(grid_w - w, rx))
        ry = max(0, min(grid_h - h, ry))
        return rx, ry, w, h

    for j, (_, row) in enumerate(rows.iterrows()):
        total = np.zeros((grid_h, grid_w), dtype=np.int16)
        have_gt = False
        for gc in gt_cols:
            if gc in row and row[gc] is not None and not (isinstance(row[gc], float) and np.isnan(row[gc])):
                x,y,w,h = (row[gc] if len(row[gc])==4 else tuple(row[gc])[:4])
                rx,ry,rw,rh = xywh_to_rect(x,y,w,h)
                total[ry:ry+rh, rx:rx+rw] += 1
                have_gt = True

        pred_mask = np.zeros((grid_h, grid_w), dtype=bool)
        for pc in pred_cols:
            if pc in row and row[pc] is not None and not (isinstance(row[pc], float) and np.isnan(row[pc])):
                x,y,w,h = (row[pc] if len(row[pc])==4 else tuple(row[pc])[:4])
                rx,ry,rw,rh = xywh_to_rect(x,y,w,h)
                pred_mask[ry:ry+rh, rx:rx+rw] = True

        if not have_gt or not pred_mask.any():
            ic_multi[j] = np.nan
            ic_ratio[j] = 0.0
            for t in thresholds:
                ic_flags[t][j] = 0.0
            continue

        vals = total[pred_mask]
        ic_multi[j] = vals.mean()
        ic_ratio[j] = (vals > 0).mean()
        for t in thresholds:
            ok = (ic_ratio[j] > t) if t == 0.0 else (ic_ratio[j] >= t)
            ic_flags[t][j] = float(ok)

    out = pd.DataFrame({
        ("union","ic_multi"): ic_multi,
        ("union","ic_ratio"): ic_ratio,
        **{("union", f"ic@{int(round(t*100)):03d}"): ic_flags[t] for t in thresholds}
    }, index=rows.index)

    return out

In [38]:
def _union_metrics_worker(args):
    (sub_df, gt_cols, pred_cols, grid_w, grid_h, orig_w, orig_h,
     kernel_x, kernel_y, use_bbox_size, thresholds) = args
    return union_metrics_singlepass(sub_df, gt_cols, pred_cols, grid_w, grid_h,
                                    orig_w, orig_h, kernel_x, kernel_y,
                                    use_bbox_size, thresholds)

In [39]:
def add_union_metrics_parallel(
    df,
    gt_cols=(("gt","0"),("gt","1"),("gt","2"),("gt","3"),("gt","4")),
    pred_cols=(("pred","0"),),
    grid_w=128, grid_h=128,
    orig_w=3456, orig_h=3720,
    kernel_x=20, kernel_y=12,
    use_bbox_size=False,
    thresholds=(0.0, 0.3, 0.5),
    max_workers=None, chunksize=None
):
    out_cols = [("union","ic_multi"), ("union","ic_ratio")] + \
               [("union", f"ic@{int(round(t*100)):03d}") for t in thresholds]
    for c in out_cols:
        if c not in df.columns:
            df[c] = np.nan

    idx = df.index.to_numpy()
    if chunksize is None:
        nworkers = max_workers or os.cpu_count() or 4
        chunksize = max(1000, len(idx)//(nworkers*4) or 1)
    splits = [idx[i:i+chunksize] for i in range(0, len(idx), chunksize)]

    arglist = [(df.loc[s].copy(), gt_cols, pred_cols, grid_w, grid_h,
                orig_w, orig_h, kernel_x, kernel_y, use_bbox_size, thresholds)
               for s in splits]

    results = []
    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futs = [ex.submit(_union_metrics_worker, a) for a in arglist]
        for f in as_completed(futs):
            results.append(f.result())

    out = pd.concat(results).sort_index()
    df.loc[out.index, out.columns] = out.values
    return df

In [40]:
out_vanilla = add_union_metrics_parallel(
    result_inference_vanilla,
    gt_cols = (("gt", "0"), ("gt", "1"), ("gt", "2"), ("gt", "3"), ("gt", "4")),
    pred_cols = (("pred", "0"),),
    use_bbox_size=False,
    thresholds=(0.0, 0.3, 0.5)
)

out_vanilla

meta                         gt                    \
           replay image_id                 0                 1   
0       11251.rep        0  [108, 2, 20, 12]  [108, 2, 20, 12]   
1       11251.rep        1  [108, 2, 20, 12]  [108, 2, 20, 12]   
2       11251.rep        2  [108, 2, 20, 12]  [108, 2, 20, 12]   
3       11251.rep        3  [108, 2, 20, 12]  [108, 2, 20, 12]   
4       11251.rep        4  [108, 2, 20, 12]  [108, 2, 20, 12]   
...           ...      ...               ...               ...   
387678   4664.rep    20690  [108, 2, 20, 12]  [108, 2, 20, 12]   
387679   4664.rep    20691  [108, 2, 20, 12]  [108, 2, 20, 12]   
387680   4664.rep    20692  [108, 2, 20, 12]  [108, 2, 20, 12]   
387681   4664.rep    20693  [108, 2, 20, 12]  [108, 2, 20, 12]   
387682   4664.rep    20694  [108, 2, 20, 12]  [108, 2, 20, 12]   

                                                              \
                       2                 3                 4   
0       [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
1       [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
2       [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
3       [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
4       [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
...                  ...               ...               ...   
387678  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
387679  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
387680  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
387681  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
387682  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   

                      pred    union                                
                         0 ic_multi ic_ratio ic@000 ic@030 ic@050  
0       [108, 113, 20, 12]      0.0      0.0    0.0    0.0    0.0  
1       [108, 113, 20, 12]      0.0      0.0    0.0    0.0    0.0  
2       [108, 113, 20, 12]      0.0      0.0    0.0    0.0    0.0  
3       [108, 113, 20, 12]      0.0      0.0    0.0    0.0    0.0  
4       [108, 113, 20, 12]      0.0      0.0    0.0    0.0    0.0  
...                    ...      ...      ...    ...    ...    ...  
387678    [95, 83, 20, 12]      0.0      0.0    0.0    0.0    0.0  
387679    [95, 83, 20, 12]      0.0      0.0    0.0    0.0    0.0  
387680    [95, 83, 20, 12]      0.0      0.0    0.0    0.0    0.0  
387681    [95, 83, 20, 12]      0.0      0.0    0.0    0.0    0.0  
387682    [95, 83, 20, 12]      0.0      0.0    0.0    0.0    0.0  

[387683 rows x 13 columns]

In [41]:
out_kbrs = add_union_metrics_parallel(
    result_inference_kbrs,
    gt_cols = (("gt", "0"), ("gt", "1"), ("gt", "2"), ("gt", "3"), ("gt", "4")),
    pred_cols = (("pred", "0"),),
    use_bbox_size=False,
    thresholds=(0.0, 0.3, 0.5)
)

out_kbrs                                          

meta                          gt                    \
           replay image_id                  0                 1   
0       11251.rep        0   [108, 2, 20, 12]  [108, 2, 20, 12]   
1       11251.rep        1   [108, 2, 20, 12]  [108, 2, 20, 12]   
2       11251.rep        2   [108, 2, 20, 12]  [108, 2, 20, 12]   
3       11251.rep        3   [108, 2, 20, 12]  [108, 2, 20, 12]   
4       11251.rep        4   [108, 2, 20, 12]  [108, 2, 20, 12]   
...           ...      ...                ...               ...   
452355   6219.rep    36788  [104, 48, 20, 12]    [0, 1, 20, 12]   
452356   6219.rep    36789  [104, 48, 20, 12]    [0, 1, 20, 12]   
452357   6219.rep    36790  [104, 48, 20, 12]    [0, 1, 20, 12]   
452358   6219.rep    36791  [104, 48, 20, 12]    [0, 1, 20, 12]   
452359   6219.rep    36792  [104, 48, 20, 12]    [0, 1, 20, 12]   

                                                                 \
                         2                 3                  4   
0         [108, 2, 20, 12]  [108, 2, 20, 12]   [108, 2, 20, 12]   
1         [108, 2, 20, 12]  [108, 2, 20, 12]   [108, 2, 20, 12]   
2         [108, 2, 20, 12]  [108, 2, 20, 12]   [108, 2, 20, 12]   
3         [108, 2, 20, 12]  [108, 2, 20, 12]   [108, 2, 20, 12]   
4         [108, 2, 20, 12]  [108, 2, 20, 12]   [108, 2, 20, 12]   
...                    ...               ...                ...   
452355  [100, 115, 20, 12]  [95, 51, 20, 12]  [102, 91, 20, 12]   
452356  [100, 115, 20, 12]  [95, 51, 20, 12]  [101, 90, 20, 12]   
452357  [100, 115, 20, 12]  [95, 51, 20, 12]  [100, 89, 20, 12]   
452358  [100, 115, 20, 12]  [95, 51, 20, 12]   [98, 87, 20, 12]   
452359  [100, 115, 20, 12]  [95, 51, 20, 12]   [97, 86, 20, 12]   

                    pred     union                                 
                       0  ic_multi  ic_ratio ic@000 ic@030 ic@050  
0         [0, 2, 19, 12]  0.000000  0.000000    0.0    0.0    0.0  
1         [0, 2, 19, 12]  0.000000  0.000000    0.0    0.0    0.0  
2         [0, 2, 19, 12]  0.000000  0.000000    0.0    0.0    0.0  
3         [0, 2, 19, 12]  0.000000  0.000000    0.0    0.0    0.0  
4         [0, 2, 19, 12]  0.000000  0.000000    0.0    0.0    0.0  
...                  ...       ...       ...    ...    ...    ...  
452355  [98, 86, 20, 12]  0.466667  0.466667    1.0    1.0    0.0  
452356  [98, 86, 20, 12]  0.566667  0.566667    1.0    1.0    1.0  
452357  [98, 86, 20, 12]  0.675000  0.675000    1.0    1.0    1.0  
452358  [98, 86, 20, 12]  0.916667  0.916667    1.0    1.0    1.0  
452359  [98, 86, 20, 12]  0.950000  0.950000    1.0    1.0    1.0  

[452360 rows x 13 columns]

In [46]:
result_inference_vanilla

meta                         gt                    \
           replay image_id                 0                 1   
0       11251.rep        0  [108, 2, 20, 12]  [108, 2, 20, 12]   
1       11251.rep        1  [108, 2, 20, 12]  [108, 2, 20, 12]   
2       11251.rep        2  [108, 2, 20, 12]  [108, 2, 20, 12]   
3       11251.rep        3  [108, 2, 20, 12]  [108, 2, 20, 12]   
4       11251.rep        4  [108, 2, 20, 12]  [108, 2, 20, 12]   
...           ...      ...               ...               ...   
387678   4664.rep    20690  [108, 2, 20, 12]  [108, 2, 20, 12]   
387679   4664.rep    20691  [108, 2, 20, 12]  [108, 2, 20, 12]   
387680   4664.rep    20692  [108, 2, 20, 12]  [108, 2, 20, 12]   
387681   4664.rep    20693  [108, 2, 20, 12]  [108, 2, 20, 12]   
387682   4664.rep    20694  [108, 2, 20, 12]  [108, 2, 20, 12]   

                                                              \
                       2                 3                 4   
0       [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
1       [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
2       [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
3       [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
4       [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
...                  ...               ...               ...   
387678  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
387679  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
387680  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
387681  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
387682  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   

                      pred    union                                
                         0 ic_multi ic_ratio ic@000 ic@030 ic@050  
0       [108, 113, 20, 12]      0.0      0.0    0.0    0.0    0.0  
1       [108, 113, 20, 12]      0.0      0.0    0.0    0.0    0.0  
2       [108, 113, 20, 12]      0.0      0.0    0.0    0.0    0.0  
3       [108, 113, 20, 12]      0.0      0.0    0.0    0.0    0.0  
4       [108, 113, 20, 12]      0.0      0.0    0.0    0.0    0.0  
...                    ...      ...      ...    ...    ...    ...  
387678    [95, 83, 20, 12]      0.0      0.0    0.0    0.0    0.0  
387679    [95, 83, 20, 12]      0.0      0.0    0.0    0.0    0.0  
387680    [95, 83, 20, 12]      0.0      0.0    0.0    0.0    0.0  
387681    [95, 83, 20, 12]      0.0      0.0    0.0    0.0    0.0  
387682    [95, 83, 20, 12]      0.0      0.0    0.0    0.0    0.0  

[387683 rows x 13 columns]

In [ ]:
gt_cols = result_inference_vanilla.columns[
    out_vanilla.columns.get_level_values(0).str.startswith("union")
]

g_vanilla = out_vanilla.groupby(("meta", "replay"))[gt_cols].mean()

out_per_replay_vanilla = g_vanilla.T.groupby(level=1).mean().T

out_per_replay_vanilla

,ic@000,ic@030,ic@050,ic_multi,ic_ratio
"(meta, replay)",,,,,
11251.rep,0.252755,0.223637,0.214218,0.234354,0.185674
1559.rep,0.255973,0.193768,0.152436,0.198402,0.146029
1628.rep,0.254546,0.232457,0.220992,0.252835,0.196130
1660.rep,0.459574,0.427494,0.387030,0.530466,0.351070
1725.rep,0.832032,0.808940,0.782164,1.593135,0.707757
212.rep,0.263804,0.251699,0.237116,0.267164,0.205146
2351.rep,0.321682,0.291783,0.279738,0.301747,0.243780
275.rep,0.744752,0.710218,0.670600,1.389356,0.603162
36.rep,0.429262,0.389516,0.346735,0.497695,0.310976


In [43]:
out_per_replay.describe()

,ic@000,ic@030,ic@050,ic_multi,ic_ratio
count,13.000000,13.000000,13.000000,13.000000,13.000000
mean,0.500183,0.470696,0.444327,0.767295,0.398768
std,0.245678,0.248429,0.244024,0.588716,0.221066
min,0.252755,0.193768,0.152436,0.198402,0.146029
25%,0.263804,0.251699,0.237116,0.267164,0.205146
50%,0.429262,0.389516,0.346735,0.497695,0.310976
75%,0.762503,0.737086,0.711814,1.389356,0.641046
max,0.832032,0.808940,0.782164,1.593135,0.707757


In [44]:
gt_cols = result_inference_kbrs.columns[
    out_kbrs.columns.get_level_values(0).str.startswith("union")
]

g = out_kbrs.groupby(("meta", "replay"))[gt_cols].mean()

out_per_replay = g.T.groupby(level=1).mean().T

out_per_replay

,ic@000,ic@030,ic@050,ic_multi,ic_ratio
"(meta, replay)",,,,,
11251.rep,0.925148,0.909745,0.884301,1.879618,0.816681
1559.rep,0.882039,0.844620,0.800380,1.434829,0.725242
1628.rep,0.825805,0.797836,0.758895,1.237885,0.686046
1660.rep,0.847064,0.822793,0.782788,1.575333,0.716305
1725.rep,0.836541,0.817242,0.779470,1.654330,0.725188
212.rep,0.824650,0.786599,0.756265,1.406372,0.689009
2351.rep,0.843138,0.807647,0.772263,1.523945,0.705352
275.rep,0.785539,0.757258,0.723234,1.535612,0.660040
36.rep,0.891917,0.874550,0.855206,1.924362,0.791322


In [45]:
out_per_replay.describe()

,ic@000,ic@030,ic@050,ic_multi,ic_ratio
count,15.000000,15.000000,15.000000,15.000000,15.000000
mean,0.854207,0.829811,0.799152,1.629768,0.732804
std,0.040202,0.044382,0.048804,0.206795,0.048460
min,0.785539,0.757258,0.723234,1.237885,0.660040
25%,0.831173,0.802625,0.766851,1.529778,0.702842
50%,0.843138,0.817242,0.782788,1.575333,0.720829
75%,0.884338,0.858008,0.833369,1.787665,0.765977
max,0.925148,0.909745,0.884328,1.932637,0.816681


In [ ]:
out_per_replay.loc[['36.rep', '212.rep', '438.rep', '522.rep', '1660.rep']].describe()

In [ ]:
def compute_total_intersection_percent(out: pd.DataFrame):
    if ("meta", "replay") not in out.columns:
        if out.index.names and "meta" in out.index.names and "replay" in out.index.names:
            out = out.reset_index()
        else:
            raise KeyError("('meta','replay')가 컬럼/인덱스 어디에도 없습니다.")

    if ("union", "ic_ratio") not in out.columns:
        raise KeyError("('union','ic_ratio') 컬럼이 없습니다. add_union_metrics_parallel 결과를 확인하세요.")

    overall_total = float(out[("union", "ic_ratio")].mean())

    g = out.groupby(("meta", "replay"))[[("union", "ic_ratio")]].mean()
    out_per_replay = g.T.groupby(level=1).mean().T
    out_per_replay = out_per_replay.rename(columns={"ic_ratio": "total_intersection_percent"})

    return overall_total, out_per_replay

In [ ]:
# compute_total_intersection_percent(out)